# Part 1: Ready Made vs Custom Made Data

# Part 2: Find Researchers using the OpenAlex API

In [2]:
import pandas as pd
from dotenv import load_dotenv
import requests
import os
from datetime import datetime
from time import sleep
import backoff # https://pypi.org/project/backoff/
from tqdm.contrib.concurrent import thread_map
from itertools import chain
import pickle
import json
from collections import Counter # For taking the mode of a list
import Levenshtein as L

d:\6_semester\02467_CSS\02467_venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [9]:
# ==========================
#   Load the scraped names
# ==========================
with open("scraped_names.json", "r", encoding="utf-8") as f:
    scraped_names = json.load(f)

scraped_names = scraped_names[0:50]
len(scraped_names)

50

In [18]:
scraped_names

['Miriam Schirmer',
 'Julia Mendelsohn',
 'Dustin Wright',
 'Dietram A. Scheufele',
 'Ágnes Horvát',
 'Caspar van Lissa',
 'Jorge Barreras',
 'Thomas Li',
 'Chen Zhong',
 'Cate Heine',
 'Adam (Zhengzi) Zhou',
 'Paolo Turrini',
 'Elias Fernández Domingos',
 'Kristina Gligoric',
 'Cinoo Lee',
 'Tijana Zrnic',
 'Adel Daoud',
 'Connor Jerzak',
 'Mark Whiting',
 'Linnea Gandhi',
 'Amirhossein Nakhaei',
 'Duncan Watts',
 'Egor Kotov',
 'Johannes Mast',
 'Matthew A. Turner',
 'James Holland Jones',
 'Dean Eckles',
 'Kathleen Carly',
 'Brandon Stewart',
 'Lisa P. Argyle',
 'Laura Nelson',
 'Amir Goldberg',
 'Arnout van de Rijt',
 'Sarah Williams',
 'IC2S2 Program Chairs',
 'Yaomin Jiang',
 'Levin Brinkmann',
 'Anne-Marie Nussberger',
 'Ivan Soraperra',
 'JF Bonnefon',
 'Iyad Rahwan',
 'Vincent Christoph Brockers',
 'David Alexander Ehrlich',
 'Viola Priesemann',
 'Pratyusha Kalluri',
 'Maggie Harrington',
 'Esin DURMUS',
 'Kiara L. Sanchez',
 'Nay San',
 'Danny Tse']

In [4]:
# ==========================
#    Method for logging 
# ==========================
timestamp = datetime.now().strftime("%Y-%m-%d_%H-%M-%S")
LOG_PATH = f"./logs/log_{timestamp}.txt"

def log_line(identifier, logText):
    """
    identifier: something that identifies the logText preferably uniquely for debugging.
    """
    with open(LOG_PATH, "a", encoding="utf-8") as f:
        f.write(f"{identifier}\t{logText}\n")
        f.flush()


# ====================================
#         -- API backoff --
# ====================================
# Retry on all RequestExceptions, exponential backoff with jitter.
@backoff.on_exception(
    backoff.expo,
    requests.exceptions.RequestException,
    max_tries=7,                     # default infinite — better to cap
    max_time=60,                     # stop after 60 sec
)
def get_with_backoff(url, **kwargs):
    resp = requests.get(url, **kwargs)

    if resp.status_code in (429, 500, 502, 503, 504):
        log_line(resp.status_code, resp.text)
        resp.raise_for_status()

    return resp



# Method for retrieving the desired information from the API query output
def getPersonInfo(p : dict):
    oid  = (p.get("id") or None)
    sid  = oid[-11:] if len(oid) >= 11 else oid

    display_name = p.get("display_name")
    
    works_api_url = p.get("works_api_url")
    
    h_index = (p.get("summary_stats") or {}).get("h_index")
    
    works_count = p.get("works_count")
    
    # Safe mode of last_known_institutions.country_code
    insts = p.get("last_known_institutions") or []
    countries = [i.get("country_code") for i in insts if (i or {}).get("country_code")]
    country_code = Counter(countries).most_common(1)[0][0] if countries else None

    cited_by_count = p.get("cited_by_count")

    return sid, display_name, works_api_url, h_index, works_count, country_code, cited_by_count


def getMaxPerson(p : dict, maxworks : int, maxcites : int, maxPerson):

    if p["works_count"] > maxworks: # If works_count is greater than the current for the scraped name
        maxPerson = getPersonInfo(p)
        maxworks = p["works_count"]
        # print("New max person1:", maxPerson)

    elif p["works_count"] == maxworks: # If works_count is equal to the current for the scraped name (tie break)
        if p["cited_by_count"] > maxPerson[-1]:
            maxPerson = getPersonInfo(p)
            # print("New max person2:", maxPerson)

    return maxworks, maxcites, maxPerson

In [20]:
# ====================================
#          API query method 
# ====================================

def getAuthor(authorNames):
    load_dotenv()
    API_KEY = os.getenv("OPENALEX_API_KEY")

    # Query the API and place the information into a pandas dataframe

    BASE = "https://api.openalex.org/authors"

    people = {}
    
    for idx, name in enumerate(authorNames):
        params = {
            "filter": "display_name.search:" + name,
            "per_page": 200,
            "api_key": API_KEY
        }

        myreq = requests.get(BASE, params=params).json()
        r = myreq
        found_exact = False

        maxworks = 0
        maxcites = 0
        maxPers = None
        if r.get("results") != []:
            for p in r.get("results"):
                # print(p["id"][-11:], p["display_name"], p["works_count"])  # For Debugging Purposes
                log_line(name, r["meta"])
                
                alt_list = p.get("display_name_alternatives") or []
                alts = [alt.casefold() for alt in alt_list]


                
                name_cf = name.casefold()
                display_cf = p["display_name"].casefold()

                is_exact = display_cf == name_cf or name_cf in alts

                if is_exact: # If an exact name match is found in display_names
                    found_exact = True
                    maxPers = getPersonInfo(p)
                    break

                # If NO exact name match is found in display_names consider the other name options with slight variations limited by the Levenshtein distance
                elif not found_exact and not is_exact: 
                    dist = L.distance(name_cf, display_cf) # Calculate the max Levenshtein distance between possible names 
                    norm = dist / max(len(name_cf), len(display_cf)) # Check Levenshtein distance proportion to the length of the name.

                    if norm <= 0.25:
                        maxworks, maxcites, maxPers = getMaxPerson(p, maxworks, maxcites, maxPers)

            # Insert maxPers into the table
            if found_exact and maxPers is not None:
                people[name] = maxPers
            elif not found_exact and maxPers is not None:
                people[name] = maxPers
            else:
                people[name] = [None, name, None, None, None, None, None]

        else:
            people[name] = [None,name,None,None,None,None,None]
            
    return people

In [21]:
# =============================================
# Parallel processing of API queries using tqdm
# =============================================

chunks = [[n] for n in scraped_names]
all_results = thread_map(getAuthor, chunks, max_workers=8, chunksize=1, desc="Fetching")

merged = {}
for d in all_results:
    merged.update(d)

df = pd.DataFrame(merged.values(), columns=['id', 'display_name', "works_api_url", "h_index", "works_count", "country_code", "cited_by_count"])
df.to_pickle(f"./datasets/OpenAlexAPI_researchers_{timestamp}.pkl")
df

Fetching: 100%|██████████| 50/50 [00:12<00:00,  3.89it/s]


,id,display_name,works_api_url,h_index,works_count,country_code,cited_by_count
0,A5036764962,Miriam Schirmer,https://api.openalex.org/works?filter=author.i...,3.0,14.0,NaN,26.0
1,A5038833789,Julia Mendelsohn,https://api.openalex.org/works?filter=author.i...,9.0,23.0,US,314.0
2,A5091326079,Dustin Wright,https://api.openalex.org/works?filter=author.i...,7.0,26.0,DK,208.0
3,A5064753440,Dietram A. Scheufele,https://api.openalex.org/works?filter=author.i...,76.0,327.0,US,28642.0
4,A5038951661,Ágnes Horvát,https://api.openalex.org/works?filter=author.i...,1.0,5.0,NaN,2.0
5,A5069426228,Caspar J. Van Lissa,https://api.openalex.org/works?filter=author.i...,36.0,200.0,NL,6543.0
6,A5089929458,Jorge Barreras,https://api.openalex.org/works?filter=author.i...,0.0,1.0,US,0.0
7,A5016322200,Thomas Li,https://api.openalex.org/works?filter=author.i...,24.0,128.0,US,2283.0
8,A5100430399,Zhong Chen,https://api.openalex.org/works?filter=author.i...,104.0,1232.0,US,46073.0
9,A5027319334,Cate Heine,https://api.openalex.org/works?filter=author.i...,7.0,21.0,GB,239.0


# Part 3: Collect Research Articles